<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.3-pretrained-apis/notebooks/GCP_Capstone_9.3_PretrainedAPIs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.3 Pre-trained APIs — Vision, NL, Translation, Document AI
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-cloud-vision google-cloud-language google-cloud-translate google-cloud-documentai google-genai Pillow

from google.cloud import vision, language_v1, translate_v3 as translate, documentai_v1 as documentai
from google.api_core import retry, exceptions
import os

PROJECT = 'your-project-id'
LOCATION = 'us-central1'
print('Pre-trained API clients ready')


## Cell 1: Vision AI - OCR for Documents


In [ ]:
# DOCUMENT_TEXT_DETECTION for dense document OCR
vision_client = vision.ImageAnnotatorClient()

# Download a sample invoice image
import urllib.request
url = 'https://upload.wikimedia.org/wikipedia/commons/0/0b/ReceiptSwiss.jpg'
urllib.request.urlretrieve(url, 'receipt.jpg')

with open('receipt.jpg', 'rb') as f:
    image = vision.Image(content=f.read())

response = vision_client.document_text_detection(image=image)
print('=== Full Text ===')
print(response.full_text_annotation.text[:500])

# Structured hierarchy: Page -> Block -> Paragraph -> Word -> Symbol
print('\n=== Block Confidence ===')
for page in response.full_text_annotation.pages:
    for i, block in enumerate(page.blocks[:3]):
        print(f'Block {i}: confidence={block.confidence:.2%}')

# SafeSearch content moderation (free when bundled)
safe_response = vision_client.safe_search_detection(image=image)
safe = safe_response.safe_search_annotation
print(f'\n=== SafeSearch ===')
print(f'Adult: {safe.adult}, Violence: {safe.violence}')


## Cell 2: Vision AI - Cost Estimation


In [ ]:
# Cost calculator for Vision AI
def vision_cost(monthly_requests, feature='text_detection'):
    free_tier = 1000
    rates = {
        'text_detection': 1.50,
        'document_text_detection': 1.50,
        'label_detection': 1.50,
        'object_localization': 2.25,
        'safe_search_detection': 1.50,
        'face_detection': 1.50,
        'logo_detection': 1.50,
    }
    rate = rates.get(feature, 1.50)
    billable = max(0, monthly_requests - free_tier)
    cost = billable * rate / 1000
    return cost

# DocuMind: 500 invoice images/month
for requests in [100, 500, 1000, 5000, 10000, 50000]:
    cost = vision_cost(requests, 'document_text_detection')
    print(f'{requests:>6,} requests: ${cost:>7.2f}/month')


## Cell 3: NL API - Entity Extraction


In [ ]:
# Entity extraction (English only - Indian languages not supported)
language_client = language_v1.LanguageServiceClient()

text = '''Sundar Pichai announced at Google I/O in Mountain View that
Alphabet will invest $5 billion in AI infrastructure during Q3 2024.
The CEO emphasized collaboration with TSMC and Samsung.'''

document = language_v1.Document(
    content=text,
    type_=language_v1.Document.Type.PLAIN_TEXT
)

# Entity extraction
response = language_client.analyze_entities(
    document=document,
    encoding_type=language_v1.EncodingType.UTF8
)

print('=== Entities ===')
for entity in response.entities:
    type_name = language_v1.Entity.Type(entity.type_).name
    print(f'{entity.name:30} [{type_name:15}] salience={entity.salience:.3f}')

# Sentiment analysis
sent_response = language_client.analyze_sentiment(document=document)
s = sent_response.document_sentiment
print(f'\n=== Sentiment ===')
print(f'Score: {s.score:.3f}, Magnitude: {s.magnitude:.3f}')


## Cell 4: NL API - Combined Features


In [ ]:
# Multiple features in one call via annotateText
features = language_v1.AnnotateTextRequest.Features(
    extract_entities=True,
    extract_document_sentiment=True,
    classify_text=True,
)

try:
    response = language_client.annotate_text(
        request={'document': document, 'features': features,
                 'encoding_type': language_v1.EncodingType.UTF8}
    )
    
    print('=== Entities ===', len(response.entities))
    print('=== Sentiment ===', response.document_sentiment.score)
    if response.categories:
        print('=== Categories ===')
        for cat in response.categories:
            print(f'  {cat.name}: {cat.confidence:.2%}')
except Exception as e:
    print(f'Note: classify_text needs 20+ tokens. Error: {e}')


## Cell 5: Translation API - Multilingual


In [ ]:
# Translation API with Indian language support
translate_client = translate.TranslationServiceClient()
parent = f'projects/{PROJECT}/locations/global'

text_en = 'The quarterly earnings exceeded expectations by 15%.'

# Translate to multiple Indian languages
indian_langs = ['hi', 'te', 'ta', 'bn', 'kn', 'ml', 'mr', 'gu']
print('=== English to Indian Languages ===')
for lang in indian_langs:
    try:
        response = translate_client.translate_text(
            contents=[text_en],
            target_language_code=lang,
            source_language_code='en',
            parent=parent
        )
        print(f'{lang}: {response.translations[0].translated_text}')
    except Exception as e:
        print(f'{lang}: (requires auth) {type(e).__name__}')

# Language detection
try:
    detected = translate_client.detect_language(
        parent=parent,
        content='नमस्कार, आप कैसे हैं?'
    )
    for lang in detected.languages:
        print(f'Detected: {lang.language_code} ({lang.confidence:.0%})')
except Exception as e:
    print(f'Detection requires auth: {type(e).__name__}')


## Cell 6: Multi-API Error Handling with Retry


In [ ]:
# Robust retry configuration for pre-trained API pipelines
RETRY_CONFIG = retry.Retry(
    initial=1.0,
    maximum=60.0,
    multiplier=2.0,
    deadline=300.0,
    predicate=retry.if_exception_type(
        exceptions.ServiceUnavailable,    # 503
        exceptions.DeadlineExceeded,       # 408
        exceptions.ResourceExhausted,      # 429 rate limit
        exceptions.InternalServerError     # 500
    )
)

# Apply to any pre-trained API call
def safe_vision_ocr(image_bytes):
    image = vision.Image(content=image_bytes)
    try:
        response = vision_client.document_text_detection(
            image=image, retry=RETRY_CONFIG)
        return response.full_text_annotation.text
    except exceptions.InvalidArgument as e:
        # Never retry 4xx client errors (except 429)
        print(f'Invalid input: {e}')
        return None

# Test
with open('receipt.jpg', 'rb') as f:
    text = safe_vision_ocr(f.read())
print(f'Extracted {len(text) if text else 0} chars with retry protection')


## Cell 7: Pipeline Cost Comparison


In [ ]:
# Compare pre-trained pipeline vs Gemini-only
def estimate_pretrained_pipeline(n_docs, pages_per_doc=3, chars_per_page=2000):
    # Vision OCR: 1K units free, $1.50/1K after
    total_pages = n_docs * pages_per_doc
    vision_billable = max(0, total_pages - 1000)
    vision_cost = vision_billable * 1.50 / 1000
    
    # NL API entities: 5K free, $1/1K units
    nl_units = total_pages * chars_per_page / 1000
    nl_billable = max(0, nl_units - 5000)
    nl_cost = nl_billable * 1.00 / 1000
    
    # Translation API: 500K chars free, $20/M chars
    total_chars = total_pages * chars_per_page
    trans_billable = max(0, total_chars - 500000)
    trans_cost = trans_billable * 20 / 1_000_000
    
    return {
        'vision': vision_cost,
        'nl': nl_cost,
        'translation': trans_cost,
        'total': vision_cost + nl_cost + trans_cost
    }

def estimate_gemini_pipeline(n_docs, pages_per_doc=3):
    # Gemini Flash: 258 tokens/PDF page input, ~500 tokens output
    input_tokens = n_docs * pages_per_doc * 258
    output_tokens = n_docs * 500
    cost = (input_tokens * 0.30 + output_tokens * 2.50) / 1_000_000
    return cost

print(f'{'n_docs':>8} {'Pre-trained':>12} {'Gemini-only':>12}  Winner')
print('=' * 50)
for n in [100, 500, 1000, 5000, 10000]:
    pt = estimate_pretrained_pipeline(n)
    gem = estimate_gemini_pipeline(n)
    winner = 'Pre-trained' if pt['total'] < gem else 'Gemini'
    print(f'{n:>8,} ${pt["total"]:>10.2f}  ${gem:>10.2f}  {winner}')


## Cell 8: Indian Language Strategy


In [ ]:
# Recommended pipeline for Indian language documents
from google import genai
from google.genai import types

gen_client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION)

def indian_language_pipeline_v1(image_bytes):
    '''Layered: Vision OCR -> Translate -> NL API entities.'''
    # Step 1: Vision AI OCR (10 Indian languages supported)
    image = vision.Image(content=image_bytes)
    ocr_response = vision_client.document_text_detection(image=image)
    source_text = ocr_response.full_text_annotation.text
    
    # Step 2: Translate to English
    parent = f'projects/{PROJECT}/locations/global'
    trans_response = translate_client.translate_text(
        contents=[source_text], target_language_code='en',
        parent=parent)
    english_text = trans_response.translations[0].translated_text
    
    # Step 3: NL API entities (English only)
    document = language_v1.Document(
        content=english_text, type_=language_v1.Document.Type.PLAIN_TEXT)
    entities = language_client.analyze_entities(document=document)
    
    return {
        'source_text': source_text[:200],
        'english_text': english_text[:200],
        'entities': [(e.name, e.salience) for e in entities.entities[:5]]
    }

def indian_language_pipeline_v2(image_bytes):
    '''Simpler: Vision OCR -> Gemini direct (handles 100+ langs).'''
    image = vision.Image(content=image_bytes)
    ocr_response = vision_client.document_text_detection(image=image)
    source_text = ocr_response.full_text_annotation.text
    
    # Gemini handles Indian languages natively for entity extraction
    response = gen_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract entities (person, organization, location, date, amount) from this text as JSON:\n\n{source_text}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            temperature=0.1
        )
    )
    return {'source_text': source_text[:200], 'entities_json': response.text}

print('Two approaches defined:')
print('  v1: Vision -> Translation -> NL API (3 API calls)')
print('  v2: Vision -> Gemini direct (2 API calls, handles code-mixed)')


## Done!
- Vision AI OCR with DOCUMENT_TEXT_DETECTION
- NL API entities and sentiment
- Translation API with 20+ Indian languages
- Retry patterns with google.api_core.retry
- Cost comparison: pre-trained vs Gemini
- Indian language layered strategy
